In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =============================================================================
# PARTIE 1 : CHARGEMENT DES DONNÉES & PRÉPARATION
# =============================================================================

path = r"C:\Users\pc\projet\return_nasdaq.m"
path_cov = r"C:\Users\pc\projet\covariance_nasdaq.m" 

try:
    data_returns = pd.read_csv(path, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    benefice = data_returns.to_numpy()
    print("--- Loading Returns Successful ---")
    print(f"Matrix Dimensions: {benefice.shape}")
except Exception as e:
    print(f"Error loading returns: {e}")
    benefice = np.random.randn(2196, 1)

try:
    data_cov = pd.read_csv(path_cov, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    covariance = data_cov.to_numpy()
    print("--- Loading Covariance Successful ---")
    print(f"Matrix Dimensions: {covariance.shape}")
except Exception as e:
    print(f"Error loading covariance: {e}")
    covariance = np.diag(np.random.uniform(0.01, 0.05, 2196))

# Configuration des paramètres de l'univers d'actifs
nb_asset = 2196
cardinal = 300

return_vect = benefice.flatten()
covar_matrix = covariance
variability = np.sqrt(np.diag(covar_matrix))

# Définition de la cible (Actifs avec rendement supérieur à la moyenne)
seuil = np.mean(return_vect)
target = (return_vect >= seuil).astype(int)

# Préparation des features pour le MLP
X = np.column_stack((return_vect, variability))
y = target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =============================================================================
# PARTIE 2 : CONFIGURATION ET ENTRAÎNEMENT DU MLP
# =============================================================================

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

print("\n--- Entraînement du MLP ---")
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

loss, accuracy, precision, recall = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision du MLP sur le test set : {accuracy:.2f}") 

# Prédiction sur l'ensemble des actifs
X_all_scaled = scaler.transform(X)
predictions_finales = (model.predict(X_all_scaled, verbose=0) > 0.5).astype(int).flatten()

# Sélection des indices retenus par le MLP
index_vector = [i for i in range(len(predictions_finales)) if predictions_finales[i] == 1]

# Filtrage par rendement pour ne garder que le top 'cardinal' (300)
return_index_vector = return_vect[index_vector]
indices_tries = sorted(range(len(return_index_vector)), key=lambda i: return_index_vector[i], reverse=True)
top_indices = indices_tries[:cardinal]
index_vector_vector = [index_vector[i] for i in top_indices]

# Création du masque binaire final (Sélection brute sans pondération)
vecteur_portefeuille = np.zeros(nb_asset)
for i in index_vector_vector:
    vecteur_portefeuille[i] = 1

# =============================================================================
# CALCULS BASÉS UNIQUEMENT SUR LE MASQUE BINAIRE (SANS ÉQUIPONDÉRATION)
# =============================================================================

# Rendement total cumulé des 300 actifs sélectionnés
rendement_total = np.sum(vecteur_portefeuille * return_vect)

# Risque total (Variance globale de la sélection brute)
risque_total = np.dot(vecteur_portefeuille.T, np.dot(covar_matrix, vecteur_portefeuille))

print("\n" + "="*70)
print("             RÉSULTATS DU FILTRAGE MLP")
print("="*70)
print(f"Nombre d'actifs sélectionnés (Cardinal) : {len(index_vector_vector)}")
print(f"Rendement Cumulé du Sous-ensemble      : {rendement_total:.6f}")
print(f"Risque Brut du Sous-ensemble (Variance) : {risque_total:.6f}")
print("="*70)

# Affichage détaillé des 300 actifs retenus
print("\nLISTE DES 300 ACTIFS SÉLECTIONNÉS :")
print("-" * 70)
print(f"{'Rang':<6} | {'Indice Original':<17} | {'Rendement Individuel':<22} | {'Risque (Écart-type)':<20}")
print("-" * 70)

for rang, idx_actif in enumerate(index_vector_vector, 1):
    rendement_indiv = return_vect[idx_actif]
    risque_indiv = variability[idx_actif]
    print(f"{rang:<6} | {idx_actif:<17} | {rendement_indiv:<22.6f} | {risque_indiv:<20.6f}")

print("-" * 70)

--- Loading Returns Successful ---
Matrix Dimensions: (2196, 1)
--- Loading Covariance Successful ---
Matrix Dimensions: (2196, 2196)

--- Entraînement du MLP ---
Précision du MLP sur le test set : 0.99

             RÉSULTATS DU FILTRAGE MLP
Nombre d'actifs sélectionnés (Cardinal) : 300
Rendement Cumulé du Sous-ensemble      : 3.263283
Risque Brut du Sous-ensemble (Variance) : 55.474549

LISTE DES 300 ACTIFS SÉLECTIONNÉS :
----------------------------------------------------------------------
Rang   | Indice Original   | Rendement Individuel   | Risque (Écart-type) 
----------------------------------------------------------------------
1      | 896               | 0.025207               | 0.143468            
2      | 1578              | 0.022523               | 0.157083            
3      | 1933              | 0.022492               | 0.120043            
4      | 1559              | 0.020579               | 0.133365            
5      | 876               | 0.020413               | 0